# Lecture 8.3 — Your First Callback: Observability

**Design Pattern:** Logging & Monitoring  
**Callbacks used:** `before_agent_callback` and `after_agent_callback`  
**Applied to:** `spending_proposer_agent` (entry) and `plan_retriever_agent` (exit)

In this lecture we upgrade the Budget Optimizer from Section 5 with **structured observability**.  
Instead of ad-hoc `print()` statements scattered across agent instructions, we attach two callback
functions that fire automatically at the precise moment the workflow starts and the moment it ends.

Every callback in this lecture returns `None` — no skipping, no overriding.  
This makes it the cleanest possible entry point: callbacks that only *observe*.

---
**Expected output after running the final cell:**
```
[ENTRY] spending_proposer_agent | inv: abc-123 | 14:02:31 | state_keys: ['budget', 'topic']
... workflow runs ...
[EXIT]  plan_retriever_agent   | inv: abc-123 | 14:03:15 | duration: 44s | state_keys: ['budget', 'topic', 'current_plan', 'critique']
```

## ⚙️ 1. Setup: Install Libraries

Pinning the version ensures our code will always work as expected.

In [3]:
!pip install google-adk==1.29.0 -q

## 🔑 2. Authentication: Configure Your API Key

In [4]:
import os
from getpass import getpass

api_key = getpass('Enter your Google API Key: ')
os.environ['GOOGLE_API_KEY'] = api_key

print("✅ API Key configured successfully!")

Enter your Google API Key: ··········
✅ API Key configured successfully!


## 🪝 3. [NEW] Define the Observability Callbacks

This is the **only new code in this lecture**. Two functions, one for entry and one for exit.

Notice the signature of each function — this is the contract the ADK framework requires:
- It takes exactly **one argument**: a `CallbackContext` object.
- It returns `Optional[types.Content]` — but we always return `None` here.

The `CallbackContext` is the ADK's messenger. It carries three things we care about:
- `callback_context.agent_name` — which agent just fired this callback
- `callback_context.invocation_id` — a unique ID for this entire workflow run
- `callback_context.state` — a live snapshot of the session's shared memory

In [5]:
# ============================================================
# NEW CELL — Lecture 8.3
# Design Pattern: Logging & Monitoring
# ============================================================

from datetime import datetime
from typing import Optional

from google.adk.agents.callback_context import CallbackContext
from google.genai import types

# A module-level variable so log_agent_exit can calculate elapsed time.
_workflow_start_time: datetime = None


def log_agent_entry(callback_context: CallbackContext) -> Optional[types.Content]:
    """
    before_agent_callback for spending_proposer_agent.

    Fires once, right before the first LLM call in the entire workflow.
    Records the start time and prints a structured ENTRY log line.

    Returns None — the agent proceeds normally. Nothing is skipped.
    """
    global _workflow_start_time
    _workflow_start_time = datetime.now()  # Capture start time for later

    # --- Read from CallbackContext ---
    agent_name    = callback_context.agent_name      # e.g. 'spending_proposer_agent'
    invocation_id = callback_context.invocation_id   # unique UUID for this run
    state_keys    = list(callback_context.state.to_dict().keys())  # what's in memory so far
    timestamp     = _workflow_start_time.strftime("%H:%M:%S")

    # --- Structured log line ---
    print("\n" + "="*60)
    print(f"[ENTRY] {agent_name}")
    print(f"        inv        : {invocation_id}")
    print(f"        timestamp  : {timestamp}")
    print(f"        state_keys : {state_keys}")
    print("="*60)

    # IMPORTANT: returning None tells the ADK framework
    # 'I'm done observing — proceed with the agent as normal.'
    return None


def log_agent_exit(callback_context: CallbackContext) -> Optional[types.Content]:
    """
    after_agent_callback for plan_retriever_agent.

    Fires once, right after the last agent in the workflow completes.
    Calculates total elapsed time and prints a structured EXIT log line.

    Returns None — the agent's output is used unchanged. Nothing is replaced.
    """
    now        = datetime.now()
    agent_name = callback_context.agent_name
    invocation_id = callback_context.invocation_id
    state_keys = list(callback_context.state.to_dict().keys())
    timestamp  = now.strftime("%H:%M:%S")

    # Calculate duration only if log_agent_entry ran first
    if _workflow_start_time is not None:
        elapsed = (now - _workflow_start_time).seconds
        duration_str = f"{elapsed}s"
    else:
        duration_str = "n/a"

    # --- Structured log line ---
    print("\n" + "="*60)
    print(f"[EXIT]  {agent_name}")
    print(f"        inv        : {invocation_id}")
    print(f"        timestamp  : {timestamp}")
    print(f"        duration   : {duration_str}")
    print(f"        state_keys : {state_keys}")
    print("="*60 + "\n")

    # IMPORTANT: returning None tells the ADK framework
    # 'I'm done observing — use the agent's real output as-is.'
    return None

## 🛠️ 4. Define Workflow Tools

Unchanged from Section 5.

In [6]:
import json
from google.adk.tools import ToolContext

def sum_costs(costs: list[float]) -> float:
    """Calculates the sum of a list of numbers."""
    print(f"  [Tool Call] sum_costs on the list: {costs}")
    return sum(costs)

def exit_loop(tool_context: ToolContext):
    """Call this function ONLY when the plan is approved and within budget."""
    print(f"  [Tool Call] Budget approved. Terminating loop: {json.dumps(tool_context.state.to_dict())}")
    tool_context.actions.escalate = True
    return None

## 4. Create Tool Wrappers

Unchanged from Section 5.

In [7]:
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool

google_search_agent = Agent(
    name="Google_Search_agent",
    model="gemini-3-flash-preview",
    instruction="You are just a wrapper for the Google Search tool.",
    tools=[google_search]
)

google_search_tool = AgentTool(agent=google_search_agent)

## 📝 5. Create Agents

**Two lines changed from Section 5** — highlighted with `# ← 8.3 CHANGE` comments:

- `spending_proposer_agent` receives `before_agent_callback=log_agent_entry`
- `plan_retriever_agent` receives `after_agent_callback=log_agent_exit`

Everything else — instructions, tools, `output_key` values — is identical to the base notebook.

In [8]:
COMPLETION_PHRASE = "The plan is within the budget."

# Agent 1: Proposes the initial, expensive plan (runs once).
# ← 8.3 CHANGE: before_agent_callback=log_agent_entry added
spending_proposer_agent = Agent(
    name="spending_proposer_agent",
    model="gemini-3-flash-preview",
    tools=[google_search],
    instruction="""
    You are a luxury event planner. For a {{topic}}, find a high-end venue and a gourmet catering service.

    Output a JSON object with items and their estimated costs, like:
    {"venue": {"name": "The Ritz London", "cost": 10000}, "catering": {"name": "Gourmet Chefs Inc.", "cost": 5000}}
    """,
    output_key="current_plan",
    before_agent_callback=log_agent_entry,  # ← 8.3 CHANGE
)

# Agent 2 (in loop): The "Accountant" that critiques the plan.
# Unchanged from Section 5.
accountant_agent = Agent(
    name="accountant_agent",
    model="gemini-3-flash-preview",
    tools=[sum_costs],
    instruction=f"""
    You are a meticulous accountant. Your budget is {{{{budget}}}}.
    The current plan is: {{{{current_plan}}}}

    Extract the costs from the plan and use the `sum_costs` tool to get the total.
    - IF the total cost is > {{{{budget}}}}, output a critique like: "This plan is over budget by [amount]. Find a cheaper [item]."
    - ELSE, respond with the exact phrase: '{COMPLETION_PHRASE}'
    """,
    output_key="critique"
)

# Agent 3 (in loop): The "Cost Cutter" that refines the plan.
# Unchanged from Section 5.
cost_cutter_agent = Agent(
    name="cost_cutter_agent",
    model="gemini-3-flash-preview",
    tools=[google_search_tool, exit_loop],
    instruction=f"""
    You are a cost-cutting expert. You must refine a plan based on a critique.
    The critique is: {{{{critique}}}}
    The current plan is: {{{{current_plan}}}}

    - IF the critique is '{COMPLETION_PHRASE}'
        1. You MUST call the `exit_loop` tool with no arguments.
        2. After calling exit_loop, output the current plan EXACTLY as-is, character for character,
           with no modifications, no acknowledgements, no commentary, and no extra text. Do not summarize it.
           Do not rephrase it. Do not add "Budget approved" or any other text.
           Just echo {{{{current_plan}}}} verbatim.
    - ELSE, read the critique to identify the overpriced item. Use your search tool to find a cheaper alternative for that item.
      Output a new JSON object with the updated plan.
    """,
    output_key="current_plan"
)

# Agent 4: Presents the final approved plan (runs once after loop).
# ← 8.3 CHANGE: after_agent_callback=log_agent_exit added
plan_retriever_agent = Agent(
    name="plan_retriever_agent",
    model="gemini-3-flash-preview",
    instruction="""
    You are a plan finalizer. Your only job is to present the final, approved plan.
    The plan is available in the context variable `{{current_plan}}`.

    Your output must be the content of the final plan presented in a clear and easy-to-read format.
    """,
    tools=[],
    after_agent_callback=log_agent_exit,  # ← 8.3 CHANGE,
    output_key="final_presentation"
)

## 🔄 6. Assemble the Loop and Sequential Workflows

Unchanged from Section 5.

In [9]:
from google.adk.agents import SequentialAgent, LoopAgent

budget_refinement_loop = LoopAgent(
    name="budget_refinement_loop",
    sub_agents=[accountant_agent, cost_cutter_agent],
    max_iterations=3,
)

budget_optimizer_workflow = SequentialAgent(
    name="budget_optimizer_workflow",
    sub_agents=[spending_proposer_agent, budget_refinement_loop, plan_retriever_agent]
)

## 🚀 7. Build the Execution Engine

Unchanged from Section 5.

In [10]:
from IPython.display import display, Markdown

from google.adk.sessions import Session
from google.genai.types import Content, Part
from google.adk.runners import Runner

async def run_agent_query(agent: Agent, query: str, topic: str, budget: str, session: Session, user_id: str):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    # Just drain the events — don't try to extract the response from them
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user"),
            state_delta={'budget': budget, 'topic': topic, 'COMPLETION_PHRASE': COMPLETION_PHRASE}
        ):
            pass
    except Exception as e:
        final_response = f"An error occurred: {e}"
        return final_response

    # ✅ Read the final plan directly from session state — always reliable
    final_session = await session_service.get_session(
        app_name=agent.name,
        user_id=user_id,
        session_id=session.id
    )
    state_dict = final_session.state
    # print("DEBUG state keys:", list(state_dict.keys()))
    # print("DEBUG current_plan:", state_dict.get("current_plan"))
    # print("DEBUG final_presentation:", state_dict.get("final_presentation"))
    # print("DEBUG critique:", state_dict.get("critique"))
    final_response = state_dict.get("final_presentation", "No plan found.")

    print("\n" + "-"*50)
    print("✅ Final Response:")
    display(Markdown(final_response))
    print("-"*50 + "\n")

    return final_response

## ✨ 8. Initialize Session and Run the Workflow

Unchanged from Section 5. Run this cell and watch for the `[ENTRY]` and `[EXIT]` log lines
that bookend the entire workflow output.

In [11]:
from google.adk.sessions import InMemorySessionService

session_service = InMemorySessionService()
user_id = "adk_event_planner_001"

In [12]:
async def run_stateful_orchestrator():

    session = await session_service.create_session(
        app_name=budget_optimizer_workflow.name,
        user_id=user_id
    )

    budget = 15000
    topic  = "50 person AI event in New York"
    query  = f"Find a plan for {topic}"

    print(f"User: {query}\n")
    await run_agent_query(budget_optimizer_workflow, query, topic, budget, session, user_id)

await run_stateful_orchestrator()

User: Find a plan for 50 person AI event in New York


🚀 Running query for agent: 'budget_optimizer_workflow' in session: '16e3c51b-c1b7-4966-9f7f-817cbf787f54'...

[ENTRY] spending_proposer_agent
        inv        : e-a3dc9466-bcaa-4fc9-a19a-b2c7ccb25d8b
        timestamp  : 02:42:24
        state_keys : ['budget', 'topic', 'COMPLETION_PHRASE']


  [Tool Call] sum_costs on the list: [15000, 20000]
  [Tool Call] sum_costs on the list: [3500, 3500]
  [Tool Call] Budget approved. Terminating loop: {"budget": 15000, "topic": "50 person AI event in New York", "COMPLETION_PHRASE": "The plan is within the budget.", "current_plan": "```json\n{\n  \"venue\": {\n    \"name\": \"The Farm SoHo\",\n    \"cost\": 3500\n  },\n  \"catering\": {\n    \"name\": \"Mangia NYC\",\n    \"cost\": 3500\n  }\n}\n```", "critique": "The plan is within the budget."}

[EXIT]  plan_retriever_agent
        inv        : e-a3dc9466-bcaa-4fc9-a19a-b2c7ccb25d8b
        timestamp  : 02:44:01
        duration   : 97s
        state_keys : ['budget', 'topic', 'COMPLETION_PHRASE', 'current_plan', 'critique', 'final_presentation']


--------------------------------------------------
✅ Final Response:


### Final Event Plan: 50-Person AI Event in New York

**Venue**
*   **Name:** The Farm SoHo
*   **Cost:** $3,500

**Catering**
*   **Name:** Mangia NYC
*   **Cost:** $3,500

**Total Estimated Budget:** $7,000

--------------------------------------------------

